In [0]:
%pip install nba_api --upgrade

  Attempting uninstall: nba_api
    Found existing installation: nba_api 1.5.2
    Uninstalling nba_api-1.5.2:
      Successfully uninstalled nba_api-1.5.2
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats
print(f"numpy: {np.__version__}")
print(f"nba_api import: OK")

numpy: 1.26.4
nba_api import: OK


In [0]:
import time
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from nba_api.stats.endpoints import leaguedashteamstats, teamgamelogs, leaguestandings

spark = SparkSession.builder.getOrCreate()


CONFIG = {
    "catalog": "workspace",
    "bronze_schema": "bronze_nba",
    "rate_limit_seconds": 0.6,      # stats.nba.com enforced rate limit
    "max_retries": 3,
    "retry_base_seconds": 5,        # exponential backoff: 5, 15, 45
    "seasons": [f"{y}-{str(y+1)[2:]}" for y in range(2015, 2024)],  # start with 2015-2023, expand later
}

print("Seasons to ingest:", CONFIG["seasons"])
print("Total API calls estimated:", len(CONFIG["seasons"]))
print("Estimated time (seconds):", len(CONFIG["seasons"]) * CONFIG["rate_limit_seconds"])

Seasons to ingest: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']
Total API calls estimated: 9
Estimated time (seconds): 5.3999999999999995


In [0]:
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats import endpoints
import nba_api.stats.library.http as nba_http


HEADERS = {
    "Host": "stats.nba.com",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "x-nba-stats-origin": "stats",
    "x-nba-stats-token": "true",
    "Connection": "keep-alive",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
}


nba_http.STATS_HEADERS = HEADERS


print("Retesting with browser headers...")

data, error = fetch_with_retry(
    fetch_fn=lambda: leaguedashteamstats.LeagueDashTeamStats(season="2023-24").get_data_frames()[0],
    season="2023-24"
)

if error:
    print(f"FAILED: {error}")
else:
    print(f"SUCCESS: {len(data)} teams, {len(data.columns)} columns")
    print("Sample columns:", list(data.columns[:8]))

Retesting with browser headers...
  [RETRY 1/3] Season 2023-24 failed: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
  Waiting 5s before retry...
  [RETRY 2/3] Season 2023-24 failed: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
  Waiting 15s before retry...
  [RETRY 3/3] Season 2023-24 failed: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
FAILED: All 3 retries exhausted for season 2023-24


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze_nba.raw_files")

DataFrame[]

In [0]:
spark.sql("SHOW VOLUMES IN workspace.bronze_nba").show()

+----------+-----------+
|  database|volume_name|
+----------+-----------+
|bronze_nba|  raw_files|
+----------+-----------+



In [0]:
import sqlite3
import tempfile
import shutil

local_path = "/tmp/nba.sqlite"
shutil.copy("/Volumes/workspace/bronze_nba/raw_files/nba.sqlite", local_path)


conn = sqlite3.connect(local_path)
cursor = conn.cursor()

cursor.execute("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")
tables = cursor.fetchall()

print(f"Found {len(tables)} tables/views:\n")
for name, type_ in tables:
    cursor.execute(f"SELECT COUNT(*) FROM [{name}]")
    count = cursor.fetchone()[0]
    print(f"  {type_:5s}  {name:40s}  {count:>10,} rows")

conn.close()

Found 16 tables/views:

  table  common_player_info                             3,632 rows
  table  draft_combine_stats                            1,633 rows
  table  draft_history                                  8,257 rows
  table  game                                          65,698 rows
  table  game_info                                     58,053 rows
  table  game_summary                                  58,110 rows
  table  inactive_players                             110,191 rows
  table  line_score                                    58,053 rows
  table  officials                                     70,971 rows
  table  other_stats                                   28,271 rows
  table  play_by_play                              13,592,899 rows
  table  player                                         4,815 rows
  table  team                                              30 rows
  table  team_details                                      27 rows
  table  team_history                 

In [0]:
conn = sqlite3.connect(local_path)

# Inspect the tables we care most about
key_tables = ["game", "line_score", "play_by_play", "team", "other_stats", "game_summary"]

for table in key_tables:
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info([{table}])")
    cols = cursor.fetchall()
    print(f"\n--- {table} ({len(cols)} columns) ---")
    for col in cols:
        print(f"  {col[1]:35s} {col[2]}")

conn.close()


--- game (55 columns) ---
  season_id                           TEXT
  team_id_home                        TEXT
  team_abbreviation_home              TEXT
  team_name_home                      TEXT
  game_id                             TEXT
  game_date                           TIMESTAMP
  matchup_home                        TEXT
  wl_home                             TEXT
  min                                 INTEGER
  fgm_home                            REAL
  fga_home                            REAL
  fg_pct_home                         REAL
  fg3m_home                           REAL
  fg3a_home                           REAL
  fg3_pct_home                        REAL
  ftm_home                            REAL
  fta_home                            REAL
  ft_pct_home                         REAL
  oreb_home                           REAL
  dreb_home                           REAL
  reb_home                            REAL
  ast_home                            REAL
  stl_home         

In [0]:
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

local_path = "/tmp/nba.sqlite"

TABLES = [
    "common_player_info", "draft_combine_stats", "draft_history",
    "game", "game_info", "game_summary", "inactive_players",
    "line_score", "officials", "other_stats", "play_by_play",
    "player", "team", "team_details", "team_history", "team_info_common"
]

# play_by_play is 13.6M rows so we have to read in chunks to avoid OOM
CHUNK_SIZE = 500_000

ingestion_log = []  

conn = sqlite3.connect(local_path)

for table in TABLES:
    print(f"\nIngesting: {table}")
    target = f"workspace.bronze_nba.{table}"
    
    try:
        if table == "play_by_play":
            # Stream in chunks -- 13.6M rows won't fit in pandas memory at once
            chunk_iter = pd.read_sql_query(
                f"SELECT * FROM [{table}]",
                conn,
                chunksize=CHUNK_SIZE
            )
            
            first_chunk = True
            total_rows = 0
            
            for chunk in chunk_iter:
                
                chunk["_ingested_at"] = datetime.now().isoformat()
                chunk["_source"] = "nba_sqlite"
                
                sdf = spark.createDataFrame(chunk.astype(str))  
                
                write_mode = "overwrite" if first_chunk else "append"
                sdf.write.format("delta").mode(write_mode).saveAsTable(target)
                
                total_rows += len(chunk)
                first_chunk = False
                print(f"  ...{total_rows:,} rows written")
        
        else:
            
            df = pd.read_sql_query(f"SELECT * FROM [{table}]", conn)
            df["_ingested_at"] = datetime.now().isoformat()
            df["_source"] = "nba_sqlite"
            
            
            sdf = spark.createDataFrame(df.astype(str))
            sdf.write.format("delta").mode("overwrite").saveAsTable(target)
            
            print(f"  {len(df):,} rows -> {target}")
        
        ingestion_log.append({"table": table, "status": "SUCCESS"})
    
    except Exception as e:
        print(f"  FAILED: {e}")
        ingestion_log.append({"table": table, "status": "FAILED", "error": str(e)})

conn.close()


print("\n\n=== INGESTION SUMMARY ===")
for entry in ingestion_log:
    status = entry["status"]
    print(f"  {status:8s}  {entry['table']}")


Ingesting: common_player_info
  3,632 rows -> workspace.bronze_nba.common_player_info

Ingesting: draft_combine_stats
  1,633 rows -> workspace.bronze_nba.draft_combine_stats

Ingesting: draft_history
  8,257 rows -> workspace.bronze_nba.draft_history

Ingesting: game
  65,698 rows -> workspace.bronze_nba.game

Ingesting: game_info
  58,053 rows -> workspace.bronze_nba.game_info

Ingesting: game_summary
  58,110 rows -> workspace.bronze_nba.game_summary

Ingesting: inactive_players
  110,191 rows -> workspace.bronze_nba.inactive_players

Ingesting: line_score
  58,053 rows -> workspace.bronze_nba.line_score

Ingesting: officials
  70,971 rows -> workspace.bronze_nba.officials

Ingesting: other_stats
  28,271 rows -> workspace.bronze_nba.other_stats

Ingesting: play_by_play
  ...500,000 rows written
  ...1,000,000 rows written
  ...1,500,000 rows written
  ...2,000,000 rows written
  ...2,500,000 rows written
  ...3,000,000 rows written
  ...3,500,000 rows written
  ...4,000,000 rows w

In [0]:
tables = [
    "common_player_info", "draft_combine_stats", "draft_history",
    "game", "game_info", "game_summary", "inactive_players",
    "line_score", "officials", "other_stats", "play_by_play",
    "player", "team", "team_details", "team_history"
]

print(f"{'table':35s} {'rows':>12s}")
print("-" * 50)

for table in tables:
    count = spark.table(f"workspace.bronze_nba.{table}").count()
    print(f"{table:35s} {count:>12,}")

table                                       rows
--------------------------------------------------
common_player_info                         3,632
draft_combine_stats                        1,633
draft_history                              8,257
game                                      65,698
game_info                                 58,053
game_summary                              58,110
inactive_players                         110,191
line_score                                58,053
officials                                 70,971
other_stats                               28,271
play_by_play                          13,592,899
player                                     4,815
team                                          30
team_details                                  27
team_history                                  50
